In [0]:
from pyspark.sql.functions import *
from pyspark.sql import Window

inventory_risk = spark.table(
    "agentdb.intelligence.inventory_risk"
)

supplier_risk = spark.table(
    "agentdb.intelligence.supplier_risk"
)

purchase_orders = spark.table(
    "agentdb.gold.fact_purchase_orders"
)

In [0]:
risk_dataset = (

    inventory_risk.alias("i")

    .join(
        purchase_orders.alias("po"),
        "product_key",
        "left"
    )

    .join(
        supplier_risk.alias("s"),
        "supplier_key",
        "left"
    )
)

In [0]:
risk_dataset = (

    risk_dataset

    .withColumn(

        "recommended_order_qty",

        greatest(
            lit(0),

            col("forecast_30d")
            +
            col("safety_stock")
            -
            col("inventory_qty")
        )
    )
)

In [0]:
risk_dataset = (

    risk_dataset

    .withColumn(

        "urgency_score",

        greatest(

            lit(0),

            least(

                lit(100),

                lit(100)
                -
                col(
                    "projected_days_to_stockout"
                )
            )
        )
    )
)

In [0]:
risk_dataset = (

    risk_dataset

    .withColumn(

        "recommendation_type",

        when(

            col(
                "projected_days_to_stockout"
            ) < 7,

            "EXPEDITE_PO"

        )

        .when(

            col(
                "s.risk_level"
            ).isin(
                "HIGH",
                "CRITICAL"
            ),

            "SUPPLIER_ALERT"
        )

        .when(

            col(
                "projected_days_to_stockout"
            ) < 30,

            "REORDER"
        )

        .otherwise(
            "NO_ACTION"
        )
    )
)

In [0]:
risk_dataset = (

    risk_dataset

    .withColumn(

        "recommendation_reason",

        when(

            col(
                "recommendation_type"
            ) == "EXPEDITE_PO",

            "Projected stockout before replenishment lead time"

        )

        .when(

            col(
                "recommendation_type"
            ) == "SUPPLIER_ALERT",

            "Supplier risk exceeds acceptable threshold"

        )

        .when(

            col(
                "recommendation_type"
            ) == "REORDER",

            "Inventory projected to fall below target coverage"

        )

        .otherwise(
            "Inventory healthy"
        )
    )
)

In [0]:
risk_dataset = (

    risk_dataset

    .withColumn(
        "recommendation_status",
        lit("OPEN")
    )
)

In [0]:
recommendations = (

    risk_dataset

    .select(

        "recommendation_type",

        "product_key",
        "store_key",
        "supplier_key",

        "recommendation_reason",

        "urgency_score",

        "recommendation_status"

    )

    .withColumnRenamed(
        "recommendation_reason",
        "recommendation_text"
    )

    .withColumn(
        "generated_at",
        current_timestamp()
    )
)
(
    recommendations

    .write

    .mode("overwrite")

    .saveAsTable(
        "agentdb.intelligence.recommendation_registry"
    )
)

In [0]:
display(

    recommendations

    .groupBy(
        "recommendation_type"
    )

    .count()
)